# 🧠 NS-ARC Deep Transformer Scaling Experiment

**Three models — 32 / 64 / 128 layer Attention Residual Transformers in every module.**

| Model | Encoder Depth | World Model | MCTS Sims |
|-------|:---:|:---:|:---:|
| `NSARC-32`  | 4  | **32**  | 25  |
| `NSARC-64`  | 8  | **64**  | 50  |
| `NSARC-128` | 12 | **128** | 100 |

## Cell 1 — Bootstrap (Kaggle / Colab)

In [ ]:
# ── Run this cell first on Kaggle / Colab ────────────────────────────
import subprocess, sys, os

def run(cmd): subprocess.run(cmd, shell=True, check=True)

# Install extra deps not in the base image
run('pip install -q wandb umap-learn scikit-learn tqdm')

# If running from a cloned repo, project root is already the cwd.
# On Kaggle: mount dataset as /kaggle/input/ns-arc and symlink:
# !ln -sf /kaggle/input/ns-arc .

# Clone ARC-AGI if not present
if not os.path.isdir('data/arc-agi'):
    run('git clone --quiet https://github.com/fchollet/ARC-AGI data/arc-agi')

print('✅ Bootstrap done')

## Cell 2 — Imports & Device

In [ ]:
import sys, os, time, json, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import torch, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

import wandb

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))
torch.manual_seed(42); np.random.seed(42)

SAVE_DIR = pathlib.Path('./checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

## Cell 3 — W&B Login

In [ ]:
# ── Paste your W&B API key here (from https://wandb.ai/authorize) ───
WANDB_KEY = os.environ.get('WANDB_API_KEY', '')   # or paste key directly

if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
else:
    wandb.login()   # interactive prompt

WANDB_PROJECT = 'NS-ARC-Scaling'
print(f'W&B project: {WANDB_PROJECT}')

## Cell 4 — Depth Profiles

In [ ]:
BASE_CFG = dict(
    latent_dim=128, action_dim=10, hidden_dim=256, nhead=8,
    in_channels=1, device=DEVICE, use_attn_res=True,
    ent_coef=0.01, gamma=0.99, lam=0.95,
    vocab_size=50, pb_c_init=1.25, pb_c_base=19652,
    curiosity_scale=0.5,
)

PROFILES = {
    'NSARC-32':  dict(**BASE_CFG, num_layers=32,  enc_depth=4,  num_simulations=25),
    'NSARC-64':  dict(**BASE_CFG, num_layers=64,  enc_depth=8,  num_simulations=50),
    'NSARC-128': dict(**BASE_CFG, num_layers=128, enc_depth=12, num_simulations=100),
}

for name, cfg in PROFILES.items():
    (SAVE_DIR / name).mkdir(exist_ok=True)

for n, c in PROFILES.items():
    print(f'{n}: WM={c["num_layers"]} layers, Enc={c["enc_depth"]} layers, MCTS sims={c["num_simulations"]}')

## Cell 5 — Dataset & Environment

In [ ]:
from data.arc_dataset import ARCDataset
from envs.arc_env import ARCEnvironment

ARC_PATH = 'data/arc-agi/training'
dataset = ARCDataset(ARC_PATH)
env     = ARCEnvironment(BASE_CFG)

sample = dataset.sample(4)
obs    = env.reset()
print(f'ARC tasks loaded  : {len(dataset.tasks)}')
print(f'Dataset batch     : {sample["state"].shape}')
print(f'Env state shape   : {obs["state"].shape}')

## Cell 6 — Build Models

In [ ]:
from modules.encoders   import TransformerEncoder
from modules.world_models import TransformerWorldModel
from modules.policies   import PPOPolicy
from modules.planners   import MCTSPlanner
from modules.curiosity  import RNDCuriosity
import torch.nn as nn

class ScaledTransformerEncoder(TransformerEncoder):
    """Uses enc_depth from config to override default ViT depth."""
    def __init__(self, cfg):
        super().__init__(cfg)
        d = cfg.get('enc_depth', 4)
        embed_dim = cfg.get('hidden_dim', 256)
        nhead = next(h for h in [8,4,2,1] if embed_dim % h == 0)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=embed_dim*4,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=d)
        self.to(self.device)

def build_model(name, cfg):
    mods = dict(
        encoder     = ScaledTransformerEncoder(cfg),
        world_model = TransformerWorldModel(cfg),
        policy      = PPOPolicy(cfg),
        planner     = MCTSPlanner(cfg),
        curiosity   = RNDCuriosity(cfg),
    )
    total = sum(p.numel() for m in mods.values() for p in m.parameters())
    print(f'{name}: {total/1e6:.1f}M params')
    return mods

MODELS = {name: build_model(name, cfg) for name, cfg in PROFILES.items()}

## Cell 7 — Training (with W&B logging)

In [ ]:
# ── Training hyperparameters ─────────────────────────────────────────
N_EPOCHS     = 50       # ← Increase for thorough training
BATCHES_EACH = 20       # batches per phase per epoch
BATCH_SIZE   = 16       # use 32+ on A100
LR           = 3e-4

def train_one_model(model_name, modules, cfg, n_epochs, batches, bs):
    # Init W&B run for this model
    run = wandb.init(
        project=WANDB_PROJECT,
        name=model_name,
        config={
            'model': model_name,
            'wm_depth': cfg['num_layers'],
            'enc_depth': cfg['enc_depth'],
            'mcts_sims': cfg['num_simulations'],
            'latent_dim': cfg['latent_dim'],
            'batch_size': bs,
            'lr': LR,
            'use_attn_res': cfg['use_attn_res'],
        },
        reinit=True
    )

    enc = modules['encoder']
    wm  = modules['world_model']
    pol = modules['policy']
    cur = modules['curiosity']
    all_params = [p for m in modules.values() for p in m.parameters()]
    opt = torch.optim.AdamW(all_params, lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    global_step = 0
    for epoch in range(n_epochs):
        t0 = time.time()
        epoch_metrics = defaultdict(list)

        for m in modules.values(): m.train()

        for b in range(batches):
            batch  = dataset.sample(bs)
            states = batch['state'].to(cfg['device'])
            a      = torch.randn(bs, cfg['action_dim'], device=cfg['device'])
            z_tar  = torch.randn(bs, cfg['latent_dim'], device=cfg['device'])
            r_tar  = torch.zeros(bs, device=cfg['device'])

            # ── Encoder ──────────────────────────────────────────────
            z = enc({'state': states})['latent']

            # ── World Model ──────────────────────────────────────────
            wm_out  = wm({'latent': z, 'action': a, 'target_latent': z_tar, 'target_reward': r_tar})
            wm_loss = wm.loss({'target_latent': z_tar, 'target_reward': r_tar}, wm_out)

            # ── Policy ───────────────────────────────────────────────
            pol_out  = pol({'latent': z.detach()})
            pol_loss = -pol_out['entropy']          # maximize entropy

            # ── Curiosity ────────────────────────────────────────────
            cur_out  = cur({'latent': z.detach()})
            cur_loss = cur.loss({}, cur_out)['loss']

            # ── Joint loss ───────────────────────────────────────────
            total = wm_loss['loss'] + 0.1 * pol_loss + 0.05 * cur_loss

            opt.zero_grad()
            total.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(all_params, 1.0)
            opt.step()

            m_dict = {
                'wm/loss':      wm_loss['loss'].item(),
                'wm/z_loss':    wm_loss.get('z_loss', wm_loss['loss']).item(),
                'wm/sigreg':    wm_loss.get('sigreg_loss', torch.tensor(0.0)).item(),
                'pol/entropy':  pol_out['entropy'].item(),
                'cur/rnd_loss': cur_loss.item(),
                'cur/reward':   cur_out['intrinsic_reward'].mean().item(),
                'e2e/loss':     total.item(),
                'grad_norm':    grad_norm.item(),
                'lr':           scheduler.get_last_lr()[0],
            }
            if 'attention_entropy' in wm_out:
                m_dict['wm/attn_entropy'] = wm_out['attention_entropy'].item()

            wandb.log(m_dict, step=global_step)
            for k, v in m_dict.items(): epoch_metrics[k].append(v)
            global_step += 1

        scheduler.step()
        mean = {k: np.mean(v) for k, v in epoch_metrics.items()}
        print(f'[{model_name}] Epoch {epoch+1:>3}/{n_epochs} | E2E={mean["e2e/loss"]:.4f} | WM={mean["wm/loss"]:.4f} | Ent={mean["pol/entropy"]:.3f} | {time.time()-t0:.0f}s')

    # ── Save modules ─────────────────────────────────────────────────
    for mod_name, mod in modules.items():
        p = SAVE_DIR / model_name / f'{mod_name}.pt'
        torch.save({'state_dict': mod.state_dict(), 'config': cfg}, p)
        artifact = wandb.Artifact(f'{model_name}_{mod_name}', type='model')
        artifact.add_file(str(p))
        run.log_artifact(artifact)

    print(f'\n✅ {model_name} saved. W&B run: {run.url}\n')
    run.finish()

# ── Train all three models sequentially ─────────────────────────────
for model_name, modules in MODELS.items():
    train_one_model(model_name, modules, PROFILES[model_name], N_EPOCHS, BATCHES_EACH, BATCH_SIZE)

## Cell 8 — Latent Space PCA (post-training)

In [ ]:
from sklearn.decomposition import PCA

COLORS = {'NSARC-32': '#4C9BE8', 'NSARC-64': '#E85D4A', 'NSARC-128': '#5CB85C'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Latent Space PCA — Post Training', fontsize=13)

for ax, (name, modules) in zip(axes, MODELS.items()):
    enc = modules['encoder'].eval()
    zs  = []
    with torch.no_grad():
        for _ in range(20):
            s = dataset.sample(8)['state'].to(PROFILES[name]['device'])
            zs.append(enc({'state': s})['latent'].cpu().numpy())
    Z    = np.concatenate(zs)
    pca  = PCA(n_components=2).fit(Z)
    proj = pca.transform(Z)

    ax.scatter(proj[:,0], proj[:,1], alpha=0.4, s=12, c=COLORS[name])
    ax.set_title(f'{name}  ({pca.explained_variance_ratio_.sum()*100:.1f}% var)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
path = str(SAVE_DIR / 'latent_pca.png')
fig.savefig(path, dpi=150)
wandb.log({'latent_pca': wandb.Image(path)})
plt.show()

## Cell 9 — Evaluation Rollout & Summary

In [ ]:
def evaluate(name, modules, cfg, n_ep=5):
    enc = modules['encoder'].eval()
    pol = modules['policy'].eval()
    rewards = []
    for _ in range(n_ep):
        obs = env.reset()
        total = 0.0
        for _ in range(50):
            s = torch.zeros(1,1,30,30)
            h = obs['state'].shape[0]
            w = obs['state'].shape[1]
            s[0,0,:min(h,30),:min(w,30)] = torch.tensor(obs['state'][:min(h,30),:min(w,30)])
            with torch.no_grad():
                z = enc({'state': s.to(cfg['device'])})['latent']
                a = pol({'latent': z})['action'].item()
            a_vec = np.zeros(cfg['action_dim'], dtype=np.float32)
            a_vec[int(a) % cfg['action_dim']] = 1.0
            obs, r, done, _ = env.step(a_vec)
            total += r
            if done: break
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

print(f'{'Model':<14} {'Mean Reward':>12} {'Std':>8}')
print('-'*36)
eval_run = wandb.init(project=WANDB_PROJECT, name='eval_summary', reinit=True)
for name, modules in MODELS.items():
    m, s = evaluate(name, modules, PROFILES[name])
    print(f'{name:<14} {m:>12.4f} {s:>8.4f}')
    wandb.log({f'eval/{name}/mean_reward': m, f'eval/{name}/std_reward': s})
eval_run.finish()
print('\n✅ Evaluation logged to W&B')